# E-Commerce Customer Intelligence & Churn Prediction

## Random Forest for Repeat Purchase Prediction

This notebook trains and evaluates a Random Forest model for repeat purchase prediction.

### Modeling Strategy

- Time-based train/test split
- Historical customer behavior features
- Severe class imbalance handling using SMOTE
- SMOTE applied only inside the training pipeline
- RandomizedSearchCV for hyperparameter optimization
- Recall used as the primary cross-validation scoring metric
- Final evaluation performed on an untouched test set

### Important

The latest historical snapshot is kept completely untouched during model training and hyperparameter tuning.

Customer identifiers, dates and future information are excluded from the model features.

In [1]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
PROJECT_ROOT = Path.cwd().parent

TRAIN_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "repeat_purchase_train.csv"
)

TEST_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "repeat_purchase_test.csv"
)

MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project Root:")
print(PROJECT_ROOT)

print("\nTrain file:")
print(TRAIN_FILE)

print("\nTest file:")
print(TEST_FILE)

Project Root:
d:\Data scientist\E-Commerce Customer Intelligence & Churn Prediction

Train file:
d:\Data scientist\E-Commerce Customer Intelligence & Churn Prediction\data\processed\repeat_purchase_train.csv

Test file:
d:\Data scientist\E-Commerce Customer Intelligence & Churn Prediction\data\processed\repeat_purchase_test.csv


In [3]:
train_df = pd.read_csv(
    TRAIN_FILE
)

test_df = pd.read_csv(
    TEST_FILE
)

print("Training Data Shape:")
print(train_df.shape)

print("\nTest Data Shape:")
print(test_df.shape)

Training Data Shape:
(115063, 18)

Test Data Shape:
(55907, 18)


In [4]:
display(
    train_df.head()
)

,customer_unique_id,snapshot_date,future_end_date,first_purchase_date,last_purchase_date,total_orders,total_revenue,average_order_value,recency_days,customer_lifetime_days,repeat_customer,purchase_frequency,mean_purchase_gap_days,median_purchase_gap_days,max_purchase_gap_days,purchase_gap_count,observation_window_days,repeat_purchase
0,ffff371b4d645b6ecea244b27531430a,2017-05-01 15:00:37,2017-10-28 15:00:37,2017-02-07 15:49:16,2017-02-07 15:49:16,1,112.46,112.46,82.966215,1.0,0,30.0,NaN,NaN,NaN,NaN,180,0
1,5353fecd3b6270bcef1daae093bb6e33,2017-05-01 15:00:37,2017-10-28 15:00:37,2017-04-25 18:03:03,2017-04-25 18:03:03,1,47.90,47.90,5.873310,1.0,0,30.0,NaN,NaN,NaN,NaN,180,0
2,535d35b2288cbdeb6a7ba74ca2173213,2017-05-01 15:00:37,2017-10-28 15:00:37,2017-02-08 14:36:40,2017-02-08 14:36:40,1,194.01,194.01,82.016632,1.0,0,30.0,NaN,NaN,NaN,NaN,180,0
3,536a45120c7e443ad9d244872e8e06a4,2017-05-01 15:00:37,2017-10-28 15:00:37,2017-04-03 12:13:04,2017-04-03 12:13:04,1,21.87,21.87,28.116354,1.0,0,30.0,NaN,NaN,NaN,NaN,180,0
4,53730ba9200fe3371560f32ee5b17424,2017-05-01 15:00:37,2017-10-28 15:00:37,2017-02-14 22:09:54,2017-02-14 22:09:54,1,41.70,41.70,75.701887,1.0,0,30.0,NaN,NaN,NaN,NaN,180,0


In [5]:
print("TRAINING DATA INFORMATION")
print("=" * 60)

train_df.info()

TRAINING DATA INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 115063 entries, 0 to 115062
Data columns (total 18 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   customer_unique_id        115063 non-null  str    
 1   snapshot_date             115063 non-null  str    
 2   future_end_date           115063 non-null  str    
 3   first_purchase_date       115063 non-null  str    
 4   last_purchase_date        115063 non-null  str    
 5   total_orders              115063 non-null  int64  
 6   total_revenue             115063 non-null  float64
 7   average_order_value       115063 non-null  float64
 8   recency_days              115063 non-null  float64
 9   customer_lifetime_days    115063 non-null  float64
 10  repeat_customer           115063 non-null  int64  
 11  purchase_frequency        115063 non-null  float64
 12  mean_purchase_gap_days    2983 non-null    float64
 13  median_purchase_gap_days  298

In [6]:
print("TEST DATA INFORMATION")
print("=" * 60)

test_df.info()

TEST DATA INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 55907 entries, 0 to 55906
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_unique_id        55907 non-null  str    
 1   snapshot_date             55907 non-null  str    
 2   future_end_date           55907 non-null  str    
 3   first_purchase_date       55907 non-null  str    
 4   last_purchase_date        55907 non-null  str    
 5   total_orders              55907 non-null  int64  
 6   total_revenue             55907 non-null  float64
 7   average_order_value       55907 non-null  float64
 8   recency_days              55907 non-null  float64
 9   customer_lifetime_days    55907 non-null  float64
 10  repeat_customer           55907 non-null  int64  
 11  purchase_frequency        55907 non-null  float64
 12  mean_purchase_gap_days    1636 non-null   float64
 13  median_purchase_gap_days  1636 non-null   float64


In [7]:
TARGET = "repeat_purchase"

print("TRAINING TARGET DISTRIBUTION")
print("=" * 60)

print(
    train_df[TARGET]
    .value_counts()
    .sort_index()
)

print("\nTRAINING TARGET PERCENTAGE")
print("=" * 60)

print(
    train_df[TARGET]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

TRAINING TARGET DISTRIBUTION
repeat_purchase
0    113393
1      1670
Name: count, dtype: int64

TRAINING TARGET PERCENTAGE
repeat_purchase
0    98.55
1     1.45
Name: proportion, dtype: float64


In [8]:
print("TEST TARGET DISTRIBUTION")
print("=" * 60)

print(
    test_df[TARGET]
    .value_counts()
    .sort_index()
)

print("\nTEST TARGET PERCENTAGE")
print("=" * 60)

print(
    test_df[TARGET]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


TEST TARGET DISTRIBUTION
repeat_purchase
0    55252
1      655
Name: count, dtype: int64

TEST TARGET PERCENTAGE
repeat_purchase
0    98.83
1     1.17
Name: proportion, dtype: float64


In [9]:
train_dates = pd.to_datetime(
    train_df["snapshot_date"]
).dt.date.unique()

test_dates = pd.to_datetime(
    test_df["snapshot_date"]
).dt.date.unique()

print("Training snapshots:")
for date in train_dates:
    print(date)

print("\nTest snapshot:")
for date in test_dates:
    print(date)

Training snapshots:
2017-05-01
2017-06-30
2017-08-29
2017-11-02
2018-01-01

Test snapshot:
2018-03-02


In [10]:
FORBIDDEN_COLUMNS = [
    "customer_unique_id",
    "snapshot_date",
    "future_end_date",
    "first_purchase_date",
    "last_purchase_date",
    "future_purchase_count",
    "churn_label",
    "repeat_purchase"
]

print("FORBIDDEN MODEL COLUMNS")
print("=" * 60)

for column in FORBIDDEN_COLUMNS:
    print("-", column)

FORBIDDEN MODEL COLUMNS
- customer_unique_id
- snapshot_date
- future_end_date
- first_purchase_date
- last_purchase_date
- future_purchase_count
- churn_label
- repeat_purchase


In [11]:
FEATURE_COLUMNS = [
    "total_orders",
    "total_revenue",
    "average_order_value",
    "recency_days",
    "customer_lifetime_days",
    "repeat_customer",
    "purchase_frequency",
    "mean_purchase_gap_days",
    "median_purchase_gap_days",
    "max_purchase_gap_days",
    "purchase_gap_count",
    "observation_window_days"
]

print("RANDOM FOREST MODEL FEATURES")
print("=" * 60)

for feature in FEATURE_COLUMNS:
    print("-", feature)

RANDOM FOREST MODEL FEATURES
- total_orders
- total_revenue
- average_order_value
- recency_days
- customer_lifetime_days
- repeat_customer
- purchase_frequency
- mean_purchase_gap_days
- median_purchase_gap_days
- max_purchase_gap_days
- purchase_gap_count
- observation_window_days


In [12]:
missing_train = [
    feature
    for feature in FEATURE_COLUMNS
    if feature not in train_df.columns
]

missing_test = [
    feature
    for feature in FEATURE_COLUMNS
    if feature not in test_df.columns
]

print("Missing training features:")
print(missing_train)

print("\nMissing testing features:")
print(missing_test)

Missing training features:
[]

Missing testing features:
[]


In [13]:
X_train = train_df[
    FEATURE_COLUMNS
].copy()

y_train = train_df[
    TARGET
].astype(int)

X_test = test_df[
    FEATURE_COLUMNS
].copy()

y_test = test_df[
    TARGET
].astype(int)

X_train = X_train.apply(
    pd.to_numeric,
    errors="coerce"
)

X_test = X_test.apply(
    pd.to_numeric,
    errors="coerce"
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (115063, 12)
y_train: (115063,)
X_test: (55907, 12)
y_test: (55907,)


In [14]:
print("TRAINING MISSING VALUES")
print("=" * 60)

display(
    X_train.isna().sum()
)

print("\nTEST MISSING VALUES")
print("=" * 60)

display(
    X_test.isna().sum()
)

TRAINING MISSING VALUES


total_orders                     0
total_revenue                    0
average_order_value              0
recency_days                     0
customer_lifetime_days           0
repeat_customer                  0
purchase_frequency               0
mean_purchase_gap_days      112080
median_purchase_gap_days    112080
max_purchase_gap_days       112080
purchase_gap_count          112080
observation_window_days          0
dtype: int64


TEST MISSING VALUES


total_orders                    0
total_revenue                   0
average_order_value             0
recency_days                    0
customer_lifetime_days          0
repeat_customer                 0
purchase_frequency              0
mean_purchase_gap_days      54271
median_purchase_gap_days    54271
max_purchase_gap_days       54271
purchase_gap_count          54271
observation_window_days         0
dtype: int64

In [15]:
train_positive = int(
    y_train.sum()
)

train_negative = int(
    (y_train == 0).sum()
)

test_positive = int(
    y_test.sum()
)

test_negative = int(
    (y_test == 0).sum()
)

print("TRAINING")
print("=" * 60)

print(
    f"Repeat purchase     : {train_positive:,}"
)

print(
    f"No repeat purchase  : {train_negative:,}"
)

print(
    f"Repeat rate         : "
    f"{y_train.mean() * 100:.2f}%"
)

print("\nTEST")
print("=" * 60)

print(
    f"Repeat purchase     : {test_positive:,}"
)

print(
    f"No repeat purchase  : {test_negative:,}"
)

print(
    f"Repeat rate         : "
    f"{y_test.mean() * 100:.2f}%"
)

TRAINING
Repeat purchase     : 1,670
No repeat purchase  : 113,393
Repeat rate         : 1.45%

TEST
Repeat purchase     : 655
No repeat purchase  : 55,252
Repeat rate         : 1.17%


In [16]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(
    "5-Fold Stratified Cross Validation configured."
)

5-Fold Stratified Cross Validation configured.


In [17]:
rf_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),

        (
            "smote",
            SMOTE(
                random_state=42
            )
        ),

        (
            "model",
            RandomForestClassifier(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Random Forest pipeline created.")

Random Forest pipeline created.


In [18]:
rf_params = {

    "smote__sampling_strategy": [
        0.02,
        0.05,
        0.10,
        0.20,
        0.30,
        0.50,
        0.75,
        1.0
    ],

    "smote__k_neighbors": [
        3,
        5,
        7
    ],

    "model__n_estimators": [
        100,
        200,
        300,
        500
    ],

    "model__max_depth": [
        None,
        5,
        10,
        15,
        20
    ],

    "model__min_samples_split": [
        2,
        5,
        10,
        20
    ],

    "model__min_samples_leaf": [
        1,
        2,
        5,
        10
    ],

    "model__max_features": [
        "sqrt",
        "log2",
        None
    ],

    "model__class_weight": [
        None,
        "balanced",
        "balanced_subsample"
    ]
}

print(
    "Random Forest search space created."
)

Random Forest search space created.


In [19]:
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,

    param_distributions=rf_params,

    n_iter=30,

    scoring="recall",

    cv=cv,

    random_state=42,

    n_jobs=-1,

    verbose=1,

    return_train_score=True
)

print(
    "RandomizedSearchCV configured."
)

print(
    "\nOptimization metric: Recall"
)

print(
    "Number of candidates: 30"
)

print(
    "Cross-validation: 5-fold StratifiedKFold"
)

RandomizedSearchCV configured.

Optimization metric: Recall
Number of candidates: 30
Cross-validation: 5-fold StratifiedKFold


In [22]:
# FAST RANDOM FOREST BASELINE

print("=" * 70)
print("TRAINING FAST RANDOM FOREST")
print("=" * 70)

rf_fast_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "smote",
            SMOTE(
                sampling_strategy=0.10,
                k_neighbors=5,
                random_state=42
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=150,
                max_depth=10,
                min_samples_split=10,
                min_samples_leaf=5,
                max_features="sqrt",
                class_weight=None,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_fast_pipeline.fit(
    X_train,
    y_train
)

print("\nTraining completed successfully.")

TRAINING FAST RANDOM FOREST

Training completed successfully.


In [23]:
rf_model = rf_fast_pipeline

print("Fast Random Forest model ready.")

Fast Random Forest model ready.


In [24]:
rf_pred = rf_model.predict(
    X_test
)

rf_prob = (
    rf_model
    .predict_proba(X_test)[:, 1]
)

print("Test predictions generated.")

Test predictions generated.


In [25]:
rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

rf_precision = precision_score(
    y_test,
    rf_pred,
    zero_division=0
)

rf_recall = recall_score(
    y_test,
    rf_pred,
    zero_division=0
)

rf_f1 = f1_score(
    y_test,
    rf_pred,
    zero_division=0
)

rf_roc_auc = roc_auc_score(
    y_test,
    rf_prob
)

rf_pr_auc = average_precision_score(
    y_test,
    rf_prob
)

print("=" * 70)
print("RANDOM FOREST TEST RESULTS")
print("=" * 70)

print(f"\nAccuracy  : {rf_accuracy:.4f}")
print(f"Precision : {rf_precision:.4f}")
print(f"Recall    : {rf_recall:.4f}")
print(f"F1 Score  : {rf_f1:.4f}")
print(f"ROC-AUC   : {rf_roc_auc:.4f}")
print(f"PR-AUC    : {rf_pr_auc:.4f}")

RANDOM FOREST TEST RESULTS

Accuracy  : 0.9873
Precision : 0.2105
Recall    : 0.0305
F1 Score  : 0.0533
ROC-AUC   : 0.5898
PR-AUC    : 0.0346


In [26]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

thresholds = np.arange(
    0.05,
    0.96,
    0.05
)

for threshold in thresholds:

    threshold_pred = (
        rf_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        threshold_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        threshold_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        threshold_pred,
        zero_division=0
    )

    predicted_positive = int(
        threshold_pred.sum()
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "predicted_repeat_customers": predicted_positive
    })

threshold_df = pd.DataFrame(
    threshold_results
)

display(
    threshold_df.round(4)
)

,threshold,precision,recall,f1,predicted_repeat_customers
0,0.05,0.0122,0.9481,0.0242,50714
1,0.10,0.0184,0.2947,0.0346,10489
2,0.15,0.0658,0.0962,0.0782,957
3,0.20,0.0919,0.0672,0.0776,479
4,0.25,0.0989,0.0534,0.0694,354
5,0.30,0.1144,0.0473,0.0670,271
6,0.35,0.1429,0.0427,0.0658,196
7,0.40,0.1724,0.0382,0.0625,145
8,0.45,0.2000,0.0351,0.0597,115
9,0.50,0.2105,0.0305,0.0533,95


In [27]:
best_f1_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

print("BEST F1 THRESHOLD")
print("=" * 60)

display(
    best_f1_row.to_frame().T.round(4)
)

BEST F1 THRESHOLD


,threshold,precision,recall,f1,predicted_repeat_customers
2,0.15,0.0658,0.0962,0.0782,957.0


In [28]:
precision_constraint = 0.20

eligible_thresholds = threshold_df[
    threshold_df["precision"] >= precision_constraint
]

if len(eligible_thresholds) > 0:

    best_recall_row = eligible_thresholds.loc[
        eligible_thresholds["recall"].idxmax()
    ]

    print(
        f"BEST RECALL WITH PRECISION >= "
        f"{precision_constraint:.0%}"
    )

    print("=" * 60)

    display(
        best_recall_row
        .to_frame()
        .T
        .round(4)
    )

else:

    print(
        "No threshold satisfies the precision constraint."
    )

BEST RECALL WITH PRECISION >= 20%


,threshold,precision,recall,f1,predicted_repeat_customers
8,0.45,0.2,0.0351,0.0597,115.0
